<a href="https://colab.research.google.com/github/opeyemiolawuwo18-droid/edge-diagnostic-classifier/blob/main/edge_diagnostic_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kagglehub torch torchvision onnx onnxruntime -q

import os
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# Dataset Download and Loading

import kagglehub
import os
from torchvision import datasets, transforms # Added: Explicitly import transforms
dataset_path = kagglehub.dataset_download("iarunava/cell-images-for-detecting-malaria")

# Point directly at the real image folders, skip the nested duplicate
base_dir = os.path.join(dataset_path, "cell_images")
print("Contents found:", os.listdir(base_dir))  # sanity check - should show the 2 real class folders + the stray one

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225]),
])

# Filter out non-image files like 'Thumbs.db'
def is_valid_image(path):
    return not path.endswith('.db') and not path.endswith('.ini')

full_dataset = datasets.ImageFolder(root=base_dir, transform=transform,
    is_valid_file=is_valid_image)

# Rebuild cleanly: filter out any samples belonging to the bad 3rd class
good_class_indices = [i for i, c in enumerate(full_dataset.classes) if c in ("Parasitized", "Uninfected")]
full_dataset.samples = [(p, l) for p, l in full_dataset.samples if l in good_class_indices]
full_dataset.targets = [l for l in full_dataset.targets if l in good_class_indices]

full_dataset.classes = ["Parasitized", "Uninfected"]

print("Classes after fix:", full_dataset.classes)
print("Total valid images:", len(full_dataset.samples))

Using Colab cache for faster access to the 'cell-images-for-detecting-malaria' dataset.
Contents found: ['Uninfected', 'Parasitized', 'cell_images']
Classes after fix: ['Parasitized', 'Uninfected']
Total valid images: 27558


In [ ]:
# Pre-trained model Loading and Adaptation

model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.last_channel, 2)
model = model.to(device)

In [ ]:
# Fine-tuning on the malaria data

import kagglehub
import os
from torchvision import datasets, transforms

# Re-define full_dataset and its dependencies, in case previous cells were not run or state was lost
dataset_path = kagglehub.dataset_download("iarunava/cell-images-for-detecting-malaria")
base_dir = os.path.join(dataset_path, "cell_images")

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225]),
])

def is_valid_image(path):
    return not path.endswith('.db') and not path.endswith('.ini')

full_dataset = datasets.ImageFolder(root=base_dir, transform=transform,
    is_valid_file=is_valid_image)

good_class_indices = [i for i, c in enumerate(full_dataset.classes) if c in ("Parasitized", "Uninfected")]
full_dataset.samples = [(p, l) for p, l in full_dataset.samples if l in good_class_indices]
full_dataset.targets = [l for l in full_dataset.targets if l in good_class_indices]
full_dataset.classes = ["Parasitized", "Uninfected"]

# Split the dataset into training, validation, and test sets
train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size])

# Create DataLoaders
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, dev=device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(dev), labels.to(dev)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

EPOCHS = 10
best_val_acc = 0
for epoch in range(EPOCHS):
    loss = train_one_epoch(model, train_loader)
    val_acc = evaluate(model, val_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} - loss: {loss:.4f} - val acc: {val_acc:.4f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pt")

model.load_state_dict(torch.load("best_model.pt"))
print(f"\nBest val accuracy: {best_val_acc:.4f}")

Using Colab cache for faster access to the 'cell-images-for-detecting-malaria' dataset.
Epoch 1/10 - loss: 0.1705 - val acc: 0.9659
Epoch 2/10 - loss: 0.0799 - val acc: 0.9659
Epoch 3/10 - loss: 0.0456 - val acc: 0.9654
Epoch 4/10 - loss: 0.0253 - val acc: 0.9637
Epoch 5/10 - loss: 0.0163 - val acc: 0.9659
Epoch 6/10 - loss: 0.0124 - val acc: 0.9681
Epoch 7/10 - loss: 0.0097 - val acc: 0.9681
Epoch 8/10 - loss: 0.0116 - val acc: 0.9683
Epoch 9/10 - loss: 0.0047 - val acc: 0.9685
Epoch 10/10 - loss: 0.0106 - val acc: 0.9683

Best val accuracy: 0.9685


In [ ]:
!pip install --upgrade protobuf -q

In [ ]:
import os
import torch

# Export to ONNX (fp32 baseline)

!pip install onnxscript -q

model_cpu = model.to("cpu")
model_cpu.eval()

dummy_input = torch.randn(1, 3, 128, 128)
torch.onnx.export(
    model_cpu, dummy_input, "malaria_model_fp32.onnx",
    input_names=["input"], output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    opset_version=13,
    dynamo=False
)
print("Exported malaria_model_fp32.onnx")

print(f"File size: {os.path.getsize("malaria_model_fp32.onnx") / (1024 * 1024):.2f} MB")

/tmp/ipykernel_14902/2040607829.py:12: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Exported malaria_model_fp32.onnx
File size: 8.47 MB


In [ ]:
# Benchmark the fp32 baseline

import onnxruntime as ort

def evaluate_onnx(model_path, loader, n_batches=None):
    session = ort.InferenceSession(model_path)
    input_name = session.get_inputs()[0].name
    correct, total, times = 0, 0, []
    for i, (images, labels) in enumerate(loader):
        if n_batches and i >= n_batches:
            break
        images_np = images.numpy()
        start = time.time()
        outputs = session.run(None, {input_name: images_np})[0]
        times.append(time.time() - start)
        preds = np.argmax(outputs, axis=1)
        correct += (preds == labels.numpy()).sum()
        total += labels.size(0)
    acc = correct / total
    avg_time_ms = np.mean(times) * 1000
    return acc, avg_time_ms

fp32_acc, fp32_time = evaluate_onnx("malaria_model_fp32.onnx", test_loader, n_batches=30)
fp32_size = os.path.getsize("malaria_model_fp32.onnx") / (1024 * 1024)

print("\n=== BASELINE (fp32 ONNX) ===")
print(f"Accuracy: {fp32_acc:.4f}")
print(f"Size: {fp32_size:.2f} MB")
print(f"Inference time: {fp32_time:.2f} ms/batch")


=== BASELINE (fp32 ONNX) ===
Accuracy: 0.9729
Size: 8.47 MB
Inference time: 308.52 ms/batch


In [ ]:
# Pre-process (required before quantization)

from onnxruntime.quantization.shape_inference import quant_pre_process

quant_pre_process(
    input_model="malaria_model_fp32.onnx",
    output_model_path="malaria_model_preprocessed.onnx"
)
print("Pre-processing complete")

Pre-processing complete


In [ ]:
# STATIC quantization with calibration

from onnxruntime.quantization import (
    quantize_static, CalibrationDataReader, QuantType, QuantFormat
)

class MalariaCalibrationReader(CalibrationDataReader):
    """Feeds a small batch of real images so the quantizer can calibrate
    proper int8 ranges for each layer - this is what static quantization
    needs that dynamic quantization skips."""
    def __init__(self, loader, input_name, n_samples=100):
        self.input_name = input_name
        self.data = []
        count = 0
        for images, _ in loader:
            for img in images:
                self.data.append(img.unsqueeze(0).numpy())
                count += 1
                if count >= n_samples:
                    break
            if count >= n_samples:
                break
        self.iterator = iter(self.data)

    def get_next(self):
        item = next(self.iterator, None)
        if item is None:
            return None
        return {self.input_name: item}

session_tmp = ort.InferenceSession("malaria_model_preprocessed.onnx")
input_name = session_tmp.get_inputs()[0].name

calib_reader = MalariaCalibrationReader(train_loader, input_name, n_samples=100)

quantize_static(
    model_input="malaria_model_preprocessed.onnx",
    model_output="malaria_model_int8.onnx",
    calibration_data_reader=calib_reader,
    quant_format=QuantFormat.QDQ,
    weight_type=QuantType.QInt8,
    activation_type=QuantType.QUInt8,
    op_types_to_quantize=["Conv", "MatMul"],
    per_channel=True,
)

print("Static quantization complete -> malaria_model_int8.onnx")

Static quantization complete -> malaria_model_int8.onnx


In [ ]:
# Benchmark the quantized model

int8_acc, int8_time = evaluate_onnx("malaria_model_int8.onnx", test_loader, n_batches=30)
int8_size = os.path.getsize("malaria_model_int8.onnx") / (1024 * 1024)

print("\n=== AFTER STATIC QUANTIZATION (int8 ONNX) ===")
print(f"Accuracy: {int8_acc:.4f}")
print(f"Size: {int8_size:.2f} MB")
print(f"Inference time: {int8_time:.2f} ms/batch")



=== AFTER STATIC QUANTIZATION (int8 ONNX) ===
Accuracy: 0.9271
Size: 2.53 MB
Inference time: 389.63 ms/batch


In [ ]:

# THE COMPARISON

print("\n=== SUMMARY: BEFORE vs AFTER ===")
print(f"Accuracy: {fp32_acc:.4f} -> {int8_acc:.4f}  (change: {(int8_acc-fp32_acc)*100:+.2f} pts)")
print(f"Size:     {fp32_size:.2f} MB -> {int8_size:.2f} MB  ({(1 - int8_size/fp32_size)*100:.1f}% smaller)")
print(f"Speed:    {fp32_time:.2f} ms -> {int8_time:.2f} ms/batch  ({fp32_time/int8_time:.2f}x faster)")


=== SUMMARY: BEFORE vs AFTER ===
Accuracy: 0.9729 -> 0.9271  (change: -4.58 pts)
Size:     8.47 MB -> 2.53 MB  (70.2% smaller)
Speed:    308.52 ms -> 389.63 ms/batch  (0.79x faster)


In [ ]:
# Export to ONNX

dummy_input = torch.randn(1, 3, 128, 128)
torch.onnx.export(
    model_cpu, dummy_input, "malaria_model.onnx",
    input_names=["input"], output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}}
)
print("\nExported to edge_malaria_model.onnx")

/tmp/ipykernel_14902/3428741245.py:4: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅

Exported to edge_malaria_model.onnx


In [ ]:
# Delete any old/stale ONNX files first
!rm -f *.onnx

# Sanity check: confirm you're working with the real trained model
print(f"Total parameters: {sum(p.numel() for p in model_cpu.parameters()):,}")
# Should show roughly 2.2-3.5 million for MobileNetV2 - if it shows way less, the model itself is wrong

Total parameters: 2,226,434
